# 📦 Notebook 01 — Data Loading and Cleaning

**Purpose:** Load all 85 Kickstarter CSV shards, deduplicate, filter to binary outcomes, perform a temporal train/validation/test split, parse JSON columns, remove leakage and junk columns, audit missing values, and save clean parquet files for downstream notebooks.

**Outputs saved to `data/`:**
- `clean_df.parquet` — full cleaned dataset
- `train_df.parquet` — temporal train split (oldest 64%)
- `val_df.parquet` — temporal validation split (middle 16%)
- `test_df.parquet` — temporal test split (newest 20%)
- `leakage_columns.json` — reference list of dropped columns

## Data Source

### Origin
The dataset is a **snapshot of Kickstarter campaign data** collected via the Kickstarter public API and made available on [WebRobots.io](https://webrobots.io/kickstarter-datasets/). The specific snapshot used here is dated **2026-02-12** (filename token: `Kickstarter_2026-02-12T03_20_22_018Z`).

### Structure
The raw data is delivered as **85 CSV shards**, each representing one page of API results paginated at the time of scraping. All shards share the same **42-column schema** (verified pre-project). Because the API paginates across an overlapping window, the same campaign may appear in multiple shards — deduplication on `id` is therefore mandatory (Step 2).

| Property | Value |
|---|---|
| Raw rows (after concat) | ~267,673 |
| Unique campaigns (after dedup) | ~209,243 |
| Campaigns with binary outcome | ~188,429 |
| Temporal span | 2009-04-25 → 2026-02-06 |
| Target variable | `success` (1 = funded, 0 = failed) |
| Class balance | ~62% success / ~38% failure |

### Key columns (pre-cleaning)
| Column | Type | Description |
|---|---|---|
| `id` | int | Unique campaign identifier |
| `name` | str | Campaign title |
| `blurb` | str | Short campaign description |
| `goal` | float | Funding goal in campaign currency |
| `launched_at` | int (Unix ts) | Campaign launch date |
| `deadline` | int (Unix ts) | Campaign end date |
| `state` | str | `successful`, `failed`, `live`, `canceled`, `suspended`, `submitted`, `started` |
| `category` | JSON str | Nested category metadata |
| `location` | JSON str | Nested location metadata |
| `staff_pick` | bool | Whether Kickstarter editorially highlighted the campaign **at launch** |
| `video` | JSON str | Video attachment metadata (None if no video) |
| `country` | str | Two-letter ISO country code |
| `pledged` | float | **LEAKAGE** — total raised (post-campaign) |
| `backers_count` | int | **LEAKAGE** — number of backers (post-campaign) |
| `spotlight` | bool | **LEAKAGE** — awarded only to successful campaigns |

### How to reproduce
1. Download the `Kickstarter_2026-02-12T03_20_22_018Z` snapshot from [https://webrobots.io/kickstarter-datasets/](https://webrobots.io/kickstarter-datasets/).
2. Unzip into `../data/Kickstarter_2026-02-12T03_20_22_018Z/`.
3. Run this notebook end-to-end.

## Setup — Imports and Paths

In [1]:
import os, sys, glob, json, re, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Paths ──────────────────────────────────────────────────────────────────
# Adjust DATA_PATH if running from a different working directory
DATA_PATH    = "../data/Kickstarter_2026-02-12T03_20_22_018Z"
OUTPUTS_PATH = "data"   # shared outputs folder for all notebooks_v2 notebooks
os.makedirs(OUTPUTS_PATH, exist_ok=True)

print(f"Data folder : {os.path.abspath(DATA_PATH)}")
print(f"Outputs path: {os.path.abspath(OUTPUTS_PATH)}")

Data folder : /Users/dacobri/Desktop/MSc Business Analytics/Classes Term 2/Artificial Intelligence 2/Final Project/Kickstarter_ML_Project/data/Kickstarter_2026-02-12T03_20_22_018Z
Outputs path: /Users/dacobri/Desktop/MSc Business Analytics/Classes Term 2/Artificial Intelligence 2/Final Project/Kickstarter_ML_Project/notebooks_v2/data


## Step 1 — Load and Merge All 85 CSV Files

We use `glob` to discover all shards and load them in a single pass.
All 85 files share the same **42-column schema** — confirmed in our pre-project analysis.

> **Note:** `low_memory=False` is required because several columns (e.g. `category`, `location`) contain mixed-type JSON strings that confuse pandas' dtype inference when chunked.

In [2]:
files = sorted(glob.glob(os.path.join(DATA_PATH, "Kickstarter*.csv")))
assert len(files) > 0, f"No CSV files found in {DATA_PATH!r}. Check DATA_PATH."
print(f"CSV files found: {len(files)}")

# Load all shards in a single list comprehension; warn on individual failures
parts = []
for f in files:
    try:
        parts.append(pd.read_csv(f, low_memory=False))
    except Exception as e:
        print(f"  [WARN] {os.path.basename(f)}: {e}")

assert parts, "All CSV files failed to load — check file integrity."
df = pd.concat(parts, ignore_index=True)
del parts  # free memory

print(f"\nShape after concat: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

CSV files found: 85

Shape after concat: (267673, 42)
Columns: ['backers_count', 'blurb', 'category', 'converted_pledged_amount', 'country', 'country_displayable_name', 'created_at', 'creator', 'currency', 'currency_symbol', 'currency_trailing_code', 'current_currency', 'deadline', 'disable_communication', 'fx_rate', 'goal', 'id', 'is_disliked', 'is_in_post_campaign_pledging_phase', 'is_launched', 'is_liked', 'is_starrable', 'launched_at', 'location', 'name', 'percent_funded', 'photo', 'pledged', 'prelaunch_activated', 'profile', 'slug', 'source_url', 'spotlight', 'staff_pick', 'state', 'state_changed_at', 'static_usd_rate', 'urls', 'usd_exchange_rate', 'usd_pledged', 'usd_type', 'video']


## Step 2 — Deduplicate on `id`

The Kickstarter API paginates results, so the same campaign can appear in multiple
shard files. We deduplicate on the unique campaign `id`.

**Expected:** ~58,000 duplicates removed, leaving ~209,000 unique campaigns.

In [3]:
n_before = len(df)
df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)
n_after = len(df)
print(f"Before deduplication : {n_before:,} rows")
print(f"After deduplication  : {n_after:,} rows")
print(f"Duplicates removed   : {n_before - n_after:,}")

Before deduplication : 267,673 rows
After deduplication  : 209,243 rows
Duplicates removed   : 58,430


## Step 3 — Filter to Binary Outcome

We keep only campaigns with a **definitive outcome**: `successful` or `failed`.  
Other states are dropped:
- `live` — campaign still running, outcome unknown
- `canceled` — creator withdrew; not a success/failure signal
- `suspended` — removed by Kickstarter; atypical cases
- `submitted` — pre-launch draft
- `started` — pre-launch state

**Expected final count:** ~188,000 rows.

In [4]:
print("All state value counts before filtering:")
print(df["state"].value_counts(dropna=False))

dropped_states = df[~df["state"].isin(["successful", "failed"])]["state"].value_counts(dropna=False)
print(f"\nDropped states:")
print(dropped_states)

df = df[df["state"].isin(["successful", "failed"])].reset_index(drop=True)
print(f"\nFinal row count: {len(df):,}")

All state value counts before filtering:
state
successful    117627
failed         70802
canceled        9134
submitted       7192
live            2995
started         1488
suspended          5
Name: count, dtype: int64

Dropped states:
state
canceled     9134
submitted    7192
live         2995
started      1488
suspended       5
Name: count, dtype: int64

Final row count: 188,429


## Step 4 — Create the Binary Target Variable

`success = 1` if the campaign reached its funding goal (`state == "successful"`).  
`success = 0` if it did not (`state == "failed"`).

We check class balance here — an important input to all modelling decisions later.

In [5]:
df["success"] = (df["state"] == "successful").astype(int)

vc = df["success"].value_counts()
print("Class distribution:")
print(f"  success = 1 (funded)  : {vc[1]:>8,}  ({vc[1]/len(df)*100:.1f}%)")
print(f"  success = 0 (failed)  : {vc[0]:>8,}  ({vc[0]/len(df)*100:.1f}%)")
print(f"  Total                 : {len(df):>8,}")
print(f"\nImbalance ratio: 1 : {vc[0]/vc[1]:.2f} (failed:funded)")

Class distribution:
  success = 1 (funded)  :  117,627  (62.4%)
  success = 0 (failed)  :   70,802  (37.6%)
  Total                 :  188,429

Imbalance ratio: 1 : 0.60 (failed:funded)


## Step 5 — Temporal Train / Validation / Test Split

### Why Temporal, Not Random?

A **random split** would be the wrong approach here for three reasons:

1. **Future data leakage**: A random split allows the model to train on campaigns from 2023
   and test on campaigns from 2015. The model would effectively see "future" patterns —
   temporal trends, macroeconomic conditions, platform algorithm changes — that would not
   be available in a real deployment setting.

2. **Mimicking real deployment**: In production, this model would be trained on historical
   campaigns and used to predict the success of *new* campaigns. A temporal split
   faithfully replicates this scenario.

3. **Distributional shift detection**: By preserving chronological order, we can later
   analyse whether the success rate or feature distributions have shifted over time —
   a critical concern for model longevity.

### Why Not Stratify?

Traditional stratified splitting (balancing the class ratio across folds) is **incompatible with temporal splitting** for this dataset. Enforcing a fixed class ratio across splits would require shuffling rows across time periods, re-introducing the very data leakage we aim to avoid. Instead:
- We **measure** class balance in each split and report it explicitly below.
- Any class imbalance within the training set is addressed at the **modelling stage** (e.g., `class_weight='balanced'`, SMOTE), not here.

### Split proportions

| Split | Proportion | Role |
|---|---|---|
| Train | 64% (oldest) | Fit model parameters |
| Validation | 16% (middle) | Tune hyperparameters, early stopping |
| Test | 20% (newest) | Final held-out evaluation — **do not touch until final report** |

We sort by `launched_at` (Unix timestamp) and slice chronologically.

In [6]:
# Convert launched_at to numeric (sometimes stored as string)
df["launched_at"] = pd.to_numeric(df["launched_at"], errors="coerce")
df = df.sort_values("launched_at").reset_index(drop=True)

# Chronological 64 / 16 / 20 split
n = len(df)
train_end_idx = int(n * 0.64)
val_end_idx   = int(n * 0.80)   # 0.64 + 0.16 = 0.80

train_df = df.iloc[:train_end_idx].copy()
val_df   = df.iloc[train_end_idx:val_end_idx].copy()
test_df  = df.iloc[val_end_idx:].copy()

def split_summary(name, split_df, width=10):
    start = pd.to_datetime(split_df["launched_at"].iloc[0],  unit="s", utc=True).strftime("%Y-%m-%d")
    end   = pd.to_datetime(split_df["launched_at"].iloc[-1], unit="s", utc=True).strftime("%Y-%m-%d")
    bal   = split_df["success"].mean() * 100
    print(f"{name:<10} : {len(split_df):>8,} rows  |  {start} → {end}  |  {bal:.1f}% success")

print("Train / Validation / Test split summary")
print("─" * 68)
split_summary("Train",      train_df)
split_summary("Validation", val_df)
split_summary("Test",       test_df)
print("─" * 68)

balances = [train_df["success"].mean()*100, val_df["success"].mean()*100, test_df["success"].mean()*100]
print(f"\u26a0  Class balance shifts across splits (train {balances[0]:.1f}% → val {balances[1]:.1f}% → test {balances[2]:.1f}%).")
print("   This is expected temporal drift, NOT an error.")
print("   Address imbalance at the modelling stage (e.g. class_weight='balanced').")

Train / Validation / Test split summary
────────────────────────────────────────────────────────────────────
Train      :  120,594 rows  |  2009-04-25 → 2022-02-11  |  59.7% success
Validation :   30,149 rows  |  2022-02-11 → 2024-06-07  |  65.1% success
Test       :   37,686 rows  |  2024-06-07 → 2026-02-06  |  69.2% success
────────────────────────────────────────────────────────────────────
⚠  Class balance shifts across splits (train 59.7% → val 65.1% → test 69.2%).
   This is expected temporal drift, NOT an error.
   Address imbalance at the modelling stage (e.g. class_weight='balanced').


## Step 6 — Parse JSON Columns

Two columns contain JSON-encoded metadata: `category` and `location`.  
We extract the most useful fields from each using a safe parser with try/except fallbacks.

- `category` → `cat_name` (e.g. "Tabletop Games"), `cat_parent_name` (e.g. "Games")
- `location` → `loc_country` (e.g. "US"), `loc_state` (e.g. "CA")

> **Important:** Parsing is applied to the **full `df`** before splitting, ensuring train/val/test receive identical feature transformations with no look-ahead.

In [7]:
def safe_json_get(val, key, default=None):
    """Extract key from JSON string with regex fallback."""
    try:
        if pd.isna(val):
            return default
        d = json.loads(str(val).replace("'", '"'))
        return d.get(key, default)
    except Exception:
        pattern = rf'"{re.escape(key)}"\s*:\s*"([^"]*)"'
        m = re.search(pattern, str(val))
        return m.group(1) if m else default

# Parse on full df (before splitting) to avoid any look-ahead
print("Parsing category column...")
df["cat_name"]        = df["category"].apply(lambda x: safe_json_get(x, "name"))
df["cat_parent_name"] = df["category"].apply(lambda x: safe_json_get(x, "parent_name"))

print("Parsing location column...")
df["loc_country"] = df["location"].apply(lambda x: safe_json_get(x, "country"))
df["loc_state"]   = df["location"].apply(lambda x: safe_json_get(x, "state"))

print(f"\ncat_name unique values     : {df['cat_name'].nunique()}")
print(f"cat_parent_name unique vals: {df['cat_parent_name'].nunique()}")
print(f"loc_country unique values  : {df['loc_country'].nunique()}")
print(f"\nSample parsed values:")
print(df[["cat_name", "cat_parent_name", "loc_country", "loc_state"]].head(5))

Parsing category column...
Parsing location column...

cat_name unique values     : 161
cat_parent_name unique vals: 15
loc_country unique values  : 207

Sample parsed values:
           cat_name cat_parent_name loc_country loc_state
0          Software      Technology        None      None
1        Journalism            None          US        NY
2           Puzzles           Games        None      None
3          Software      Technology          US        NY
4  Electronic Music           Music          US        LA


## Step 7 — Drop Leakage Columns

These columns must be **unconditionally removed** before any modelling.
They contain information that is only available **after** the campaign ends —
using them would cause catastrophic data leakage.

| Column | Why it's leakage |
|---|---|
| `pledged` | Total money raised — only known after campaign ends |
| `usd_pledged` | Same as pledged, in USD |
| `converted_pledged_amount` | Same, currency-converted |
| `backers_count` | Number of backers — only known after campaign ends |
| `percent_funded` | `pledged / goal × 100` — the answer expressed as a percentage |
| `spotlight` | **⚠️ MOST DANGEROUS**: Kickstarter awards spotlight status ONLY to successful campaigns. Pearson r = 1.0 with success. Including this would give 100% accuracy while learning nothing useful. |
| `state_changed_at` | Timestamp when outcome was recorded — post-campaign |
| `usd_exchange_rate` | Rate at settlement — post-campaign |
| `static_usd_rate` | Same |
| `fx_rate` | Same |

### Note on `staff_pick`
`staff_pick` is **NOT** leakage. Kickstarter applies this editorial flag at or before campaign launch as a promotion decision — it is observable at prediction time. It is therefore a legitimate pre-campaign feature and is retained.

In [8]:
LEAKAGE_COLS = [
    "pledged", "usd_pledged", "converted_pledged_amount",
    "backers_count", "percent_funded", "spotlight",
    "state_changed_at", "usd_exchange_rate", "static_usd_rate", "fx_rate",
]

# Verify spotlight is perfectly correlated with success (Pearson r = 1.0)
if "spotlight" in df.columns:
    sp = df["spotlight"].map({"True": 1, "False": 0, True: 1, False: 0}).fillna(0)
    corr = sp.corr(df["success"])
    print(f"spotlight ↔ success Pearson r = {corr:.4f}  ← this MUST be dropped")

# Drop leakage columns that exist in the dataframe
cols_to_drop = [c for c in LEAKAGE_COLS if c in df.columns]
df = df.drop(columns=cols_to_drop)
print(f"\nDropped {len(cols_to_drop)} leakage columns: {cols_to_drop}")
print(f"Shape after leakage drop: {df.shape}")

# Save reference list
with open(os.path.join(OUTPUTS_PATH, "leakage_columns.json"), "w") as f:
    json.dump({"leakage_columns": LEAKAGE_COLS}, f, indent=2)

spotlight ↔ success Pearson r = 1.0000  ← this MUST be dropped

Dropped 10 leakage columns: ['pledged', 'usd_pledged', 'converted_pledged_amount', 'backers_count', 'percent_funded', 'spotlight', 'state_changed_at', 'usd_exchange_rate', 'static_usd_rate', 'fx_rate']
Shape after leakage drop: (188429, 37)


## Step 8 — Drop Junk / Metadata Columns

These columns carry **no predictive signal** — they are URL links, image paths,
API metadata, redundant identifiers, or constant-value columns. Keeping them would
only add noise and slow down training.

| Column | Reason for removal |
|---|---|
| `photo`, `profile`, `urls` | Raw JSON blobs of image/URL metadata |
| `creator` | Nested JSON with creator profile (too sparse to use directly) |
| `source_url`, `slug` | URL-path identifiers, no signal |
| `currency_symbol`, `currency_trailing_code`, `current_currency` | Redundant with `currency` |
| `is_liked`, `is_disliked`, `is_starrable` | API interaction flags, not creator attributes |
| `is_in_post_campaign_pledging_phase` | Post-campaign state flag — potential leakage |
| `usd_type`, `prelaunch_activated` | Sparse flags with negligible signal |
| `category`, `location` | Raw JSON — already extracted into `cat_*` / `loc_*` columns |
| `is_launched` | **Constant column** (all values = True after Step 3 filter) — zero variance, no signal |
| `country_displayable_name` | **Redundant** with `country` (ISO code) — same information in verbose form |

In [9]:
JUNK_COLS = [
    "photo", "profile", "urls", "creator", "source_url", "slug",
    "currency_symbol", "currency_trailing_code", "current_currency",
    "is_liked", "is_disliked", "is_starrable",
    "is_in_post_campaign_pledging_phase", "usd_type", "prelaunch_activated",
    # raw JSON columns (we've extracted what we need)
    "category", "location",
    # constant column (all True after Step 3 filter — zero variance)
    "is_launched",
    # redundant with `country` (ISO code carries the same information)
    "country_displayable_name",
]

# Defensive check: report any additional constant columns not already listed
constant_cols = [c for c in df.columns if df[c].nunique() == 1 and c not in JUNK_COLS]
if constant_cols:
    print(f"[WARN] Additional constant columns found — consider adding to JUNK_COLS: {constant_cols}")

# Report which listed constants are actually present
constant_present = [c for c in JUNK_COLS if c in df.columns and df[c].nunique() == 1]
if constant_present:
    print(f"Constant columns detected (1 unique value) — dropping: {constant_present}")

cols_to_drop_junk = [c for c in JUNK_COLS if c in df.columns]
df = df.drop(columns=cols_to_drop_junk)
print(f"\nDropped {len(cols_to_drop_junk)} junk columns")
print(f"Shape after junk drop: {df.shape}")
print(f"\nRemaining columns ({len(df.columns)}):")
for col in df.columns:
    print(f"  {col}")

Constant columns detected (1 unique value) — dropping: ['current_currency', 'is_liked', 'is_disliked', 'is_starrable', 'usd_type', 'is_launched']

Dropped 19 junk columns
Shape after junk drop: (188429, 18)

Remaining columns (18):
  blurb
  country
  created_at
  currency
  deadline
  disable_communication
  goal
  id
  launched_at
  name
  staff_pick
  state
  video
  success
  cat_name
  cat_parent_name
  loc_country
  loc_state


## Step 9 — Missing Value Audit

After cleaning, we audit every remaining column for missing values.
Missingness is **often informative**:
- Missing `video` → the creator did not include a video (a meaningful choice)
- Missing `cat_name` → the category JSON could not be parsed

### Handling strategy per column

| Column | Missing | Strategy |
|---|---|---|
| `video` | ~32.9% | Create binary `has_video` feature in feat. engineering; drop raw column |
| `cat_parent_name` | ~2.2% | Impute with `"Unknown"` — preserve category structure |
| `loc_state` | ~0.1% | Impute with `"Unknown"` — non-US campaigns legitimately lack a state |
| `loc_country` | ~0.08% | Impute with `country` column value (same information, rarely differs) |
| `blurb` | 4 rows | Impute with empty string `""` — treat as no description given |

In [10]:
audit = pd.DataFrame({
    "dtype": df.dtypes,
    "n_missing": df.isnull().sum(),
    "pct_missing": (df.isnull().mean() * 100).round(2),
    "n_unique": df.nunique(),
}).sort_values("pct_missing", ascending=False)

print("Missing value audit:")
print(audit.to_string())

# ── Apply immediate fixes ────────────────────────────────────────────────
print("\nApplying immediate missing value fixes:")

# blurb: 4 missing → empty string (no description given by creator)
n = df["blurb"].isnull().sum()
df["blurb"] = df["blurb"].fillna("")
print(f"  [blurb]  {n} rows → filled with empty string")

# cat_parent_name: ~2% missing → 'Unknown' preserves the category structure
n = df["cat_parent_name"].isnull().sum()
df["cat_parent_name"] = df["cat_parent_name"].fillna("Unknown")
print(f"  [cat_parent_name]  {n:,} rows → filled with 'Unknown'")

# loc_state: missing for non-US campaigns — 'Unknown' is appropriate
n = df["loc_state"].isnull().sum()
df["loc_state"] = df["loc_state"].fillna("Unknown")
print(f"  [loc_state]  {n:,} rows → filled with 'Unknown'")

# loc_country: backfill from the `country` ISO column
n = df["loc_country"].isnull().sum()
df["loc_country"] = df["loc_country"].fillna(df["country"])
print(f"  [loc_country]  {n:,} rows → backfilled from country column")

# video: intentionally deferred — missingness IS the signal (has_video flag)
print(f"  [video]  {df['video'].isnull().sum():,} rows → deferred to feature engineering (has_video flag)")

remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print(f"\nMissing values remaining after fixes:")
print(remaining_missing)
print("(Only `video` remains missing — handled in feature engineering)")

Missing value audit:
                         dtype  n_missing  pct_missing  n_unique
video                   object      62016      32.9100    126413
cat_parent_name         object       4210       2.2300        15
loc_state               object        195       0.1000      1196
loc_country             object        146       0.0800       207
country                 object          0       0.0000        25
cat_name                object          0       0.0000       161
success                  int64          0       0.0000         2
state                   object          0       0.0000         2
staff_pick                bool          0       0.0000         2
blurb                   object          4       0.0000    185968
launched_at              int64          0       0.0000    188210
id                       int64          0       0.0000    188429
goal                   float64          0       0.0000      6268
disable_communication     bool          0       0.0000         2
dead

## Step 10 — Save Outputs

We save four parquet files and the leakage reference JSON.  
Parquet is used instead of CSV because it:
- Preserves column dtypes (no re-parsing needed)
- Is ~3–5× faster to read/write
- Compresses better for large DataFrames

> **Reminder:** The splits are rebuilt here from the final cleaned `df` to ensure all cleaning steps (missing value fixes, column drops) are reflected in all three files.

In [11]:
# Rebuild splits from the fully cleaned df
print("Rebuilding splits from final cleaned df...")
df["launched_at"] = pd.to_numeric(df["launched_at"], errors="coerce")
df = df.sort_values("launched_at").reset_index(drop=True)

n = len(df)
train_end_idx = int(n * 0.64)
val_end_idx   = int(n * 0.80)

train_df = df.iloc[:train_end_idx].copy()
val_df   = df.iloc[train_end_idx:val_end_idx].copy()
test_df  = df.iloc[val_end_idx:].copy()

# Save
clean_path = os.path.join(OUTPUTS_PATH, "clean_df.parquet")
train_path = os.path.join(OUTPUTS_PATH, "train_df.parquet")
val_path   = os.path.join(OUTPUTS_PATH, "val_df.parquet")
test_path  = os.path.join(OUTPUTS_PATH, "test_df.parquet")

df.to_parquet(clean_path, index=False)
train_df.to_parquet(train_path, index=False)
val_df.to_parquet(val_path, index=False)
test_df.to_parquet(test_path, index=False)

print(f"\nSaved clean_df  : {len(df):,} rows → {clean_path}")
print(f"Saved train_df  : {len(train_df):,} rows → {train_path}")
print(f"Saved val_df    : {len(val_df):,} rows → {val_path}")
print(f"Saved test_df   : {len(test_df):,} rows → {test_path}")
print(f"\nNotebook 01 complete. Proceed to 02_eda.ipynb")

Rebuilding splits from final cleaned df...

Saved clean_df  : 188,429 rows → data/clean_df.parquet
Saved train_df  : 120,594 rows → data/train_df.parquet
Saved val_df    : 30,149 rows → data/val_df.parquet
Saved test_df   : 37,686 rows → data/test_df.parquet

Notebook 01 complete. Proceed to 02_eda.ipynb
